In [1]:
%%writefile services.py
import os
import re
import io
import email
import time
import json
import torch
import shutil
import imaplib
import smtplib
import tempfile
import pandas as pd
import fitz  
from PIL import Image
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from email.header import decode_header
from email.mime.text import MIMEText
from hijri_converter import Hijri
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever, ParentDocumentRetriever
from langchain.storage import InMemoryStore
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from googleapiclient.discovery import build
from google.oauth2 import service_account
from ArabicOcr import arabicocr  
import gc  
import openpyxl  
import json
import re
from datetime import datetime, timedelta
from hijri_converter import Hijri, Gregorian
from googleapiclient.discovery import build
from google.oauth2 import service_account
import json
import re
import re
import json
from RulesForIntents import RULES_DIR , load_intent_rules , deduce_administrative_rule , update_intent_rules_file 


IMAP_SERVER = "imap.gmail.com"
GMAIL_EMAIL = "eng.mansour.issa@gmail.com"
APP_PASSWORD = "sugtcfmplficwqzr"
CALENDAR_ID = "eng.mansour.issa@gmail.com"
SERVICE_ACCOUNT_FILE = 'kaust-481121-7ec069937b0c.json'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SCOPES = ['https://www.googleapis.com/auth/calendar']

Overwriting services.py


In [2]:
%%writefile -a services.py

def safe_decode(payload):
    if payload is None: return ""
    for enc in ["utf-8", "iso-8859-1", "cp1256"]:
        try: return payload.decode(enc)
        except: continue
    return payload.decode("utf-8", errors="ignore")




Appending to services.py


In [3]:
%%writefile -a services.py

def html_to_text(html): return BeautifulSoup(html, "html.parser").get_text()

Appending to services.py


In [4]:
%%writefile -a services.py
SMTP_SERVER = "smtp.gmail.com"

def send_reply(to, subject, text):
    msg = MIMEText(text, "plain", "utf-8")
    msg["Subject"] = f"Re: {subject}"
    msg["From"] = GMAIL_EMAIL
    msg["To"] = to
    try:
        with smtplib.SMTP(SMTP_SERVER, 587, timeout=20) as server:
            server.starttls()
            server.login(GMAIL_EMAIL, APP_PASSWORD)
            server.sendmail(GMAIL_EMAIL, [to], msg.as_string())
        print(f"✔ تم إرسال رد إلى {to}")
    except Exception as e:
        print(f"⚠ خطأ إرسال رد: {e}")

Appending to services.py


In [5]:
%%writefile -a services.py

def check_new_emails(processor):
    """
    فحص البريد الوارد واستخراج الرسائل الجديدة مع معرفاتها (Message-ID).
    """
    import gc
    new_emails = []
    
    try:
        mail = imaplib.IMAP4_SSL(IMAP_SERVER)
        mail.login(GMAIL_EMAIL, APP_PASSWORD)
        mail.select("INBOX")
        
        since = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")
        status, msgs = mail.search(None, f'(UNSEEN SINCE "{since}")')
        msg_ids = msgs[0].split() if msgs[0] else []
        
        if msg_ids:
            print(f"📨 {len(msg_ids)} رسائل جديدة.")
        
        for msg_id in msg_ids:
            try:
                _, data = mail.fetch(msg_id, "(RFC822)")
                raw = data[0][1]
                msg = email.message_from_bytes(raw)
                
                # استخراج البيانات الأساسية
                sender = email.utils.parseaddr(msg["From"])[1]
                subject = msg["Subject"]
                message_id_header = msg.get("Message-ID", "").strip() # استخراج المعرف الفريد
                
                body = ""
                attachments = []
                
                if msg.is_multipart():
                    for part in msg.walk():
                        if part.get_content_type() == "text/plain":
                            body = safe_decode(part.get_payload(decode=True))
                        
                        if part.get('Content-Disposition') is None: continue
                        filename = part.get_filename()
                        if filename:
                            decoded = decode_header(filename)[0]
                            if isinstance(decoded[0], bytes):
                                filename = decoded[0].decode(decoded[1] or 'utf-8')
                            
                            with tempfile.NamedTemporaryFile(delete=False) as temp_file:
                                temp_file.write(part.get_payload(decode=True))
                                temp_path = temp_file.name
                            
                            os.makedirs("attachments", exist_ok=True)
                            permanent_path = os.path.join("attachments", filename)
                            shutil.copy(temp_path, permanent_path)
                            attachments.append(permanent_path)
                            os.unlink(temp_path)
                else:
                    body = safe_decode(msg.get_payload(decode=True))
                
                # نمرر الآن معرف الرسالة (message_id_header)
                new_emails.append((sender, subject, body, attachments, message_id_header))
                
                mail.store(msg_id, '+FLAGS', '\\Seen')
                gc.collect()

            except Exception as e:
                print(f"خطأ في معالجة الرسالة {msg_id}: {e}")
                continue
                
        mail.logout()
    except Exception as e:
        print(f"خطأ في الاتصال بالبريد: {e}")
    
    return new_emails

Appending to services.py


In [6]:
%%writefile -a services.py

def update_email_history(email_type, content, msg_id=None, intent=None):
    """
    حفظ الإيميل في السجل مع البيانات الوصفية (ID, Intent) لاسترجاعها لاحقاً عند التعلم.
    """
    from datetime import datetime
    date_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # بناء ترويسة غنية بالمعلومات (مخفية داخل النص لتسهيل قراءتها بالكود)
    meta_info = []
    if msg_id: meta_info.append(f"Ref-ID: {msg_id}")
    if intent: meta_info.append(f"Org-Intent: {intent}")
    
    header = f"{email_type} بتاريخ {date_str}"
    meta_str = " | ".join(meta_info)
    
    entry = f"{header}\n[{meta_str}]\n{content}\n\n"
    
    email_history_file = "email_history.txt"
    with open(email_history_file, 'a', encoding='utf-8') as f:
        f.write(entry)
    
    # الحفاظ على حجم الملف (آخر 50)
    try:
        full_content = open(email_history_file, 'r', encoding='utf-8').read()
        # نستخدم فاصل ثابت بدلاً من الاعتماد الكلي على regex
        entries = full_content.split("إيميل وارد بتاريخ")
        if len(entries) > 51: # 50 + 1 (empty start)
            kept_entries = entries[-50:]
            reconstructed = "إيميل وارد بتاريخ".join(kept_entries)
            if not reconstructed.startswith("إيميل وارد بتاريخ"):
                 reconstructed = "إيميل وارد بتاريخ" + reconstructed
            with open(email_history_file, 'w', encoding='utf-8') as f:
                f.write(reconstructed)
    except:
        pass # تجاوز أخطاء التنظيف لتجنب توقف النظام


Appending to services.py


In [7]:
%%writefile -a services.py


def create_draft(to, subject, body, reply_to_id=None):
    """
    إنشاء مسودة مع إضافة رؤوس الربط (In-Reply-To) لضمان تتبع المحادثة.
    """
    try:
        mail = imaplib.IMAP4_SSL(IMAP_SERVER)
        mail.login(GMAIL_EMAIL, APP_PASSWORD)
        
        msg = email.message.Message()
        msg['Subject'] = f"Re: {subject}" if not subject.startswith("Re:") else subject
        msg['From'] = GMAIL_EMAIL
        msg['To'] = to
        
        # الربط التقني بالإيميل الأصلي
        if reply_to_id:
            msg['In-Reply-To'] = reply_to_id
            msg['References'] = reply_to_id
            
        msg.set_payload(body, charset='utf-8')
        
        now = imaplib.Time2Internaldate(time.time())
        mail.append("[Gmail]/Drafts", '', now, msg.as_bytes()) # تأكد من اسم المجلد حسب إعداداتك
        
        mail.logout()
        print(f"✉️ تم حفظ الرد كمسودة موجهة إلى {to}")
        return True
    except Exception as e:
        print(f"⚠ خطأ أثناء إنشاء المسودة: {e}")
        return False

Appending to services.py


In [8]:
%%writefile -a services.py
def update_evaluation_ground_truth(ref_id, final_body):
    """
    تحديث ملف التقييم بالنص النهائي المرسل فعلياً (Ground Truth)
    بناءً على معرف الرسالة الأصلية (In-Reply-To).
    """
    file_path = "evaluation_data.xlsx"
    if not os.path.exists(file_path) or not ref_id:
        return

    try:
        # تنظيف المعرف من الأقواس الزاوية لضمان المطابقة
        clean_ref_id = ref_id.strip().strip('<>')
        
        df = pd.read_excel(file_path)
        
        # البحث عن الصف الذي يحتوي على هذا المعرف في عمود Message-ID
        # نستخدم apply للتأكد من تنظيف المعرفات في الإكسل أيضاً عند المقارنة
        mask = df['Message-ID'].astype(str).apply(lambda x: clean_ref_id in x)
        
        if mask.any():
            df.loc[mask, 'Ground Truth'] = final_body
            df.to_excel(file_path, index=False)
            print(f"📊 تم تحديث Ground Truth للإيميل المرتبط بـ: {clean_ref_id}")
    except Exception as e:
        print(f"⚠ خطأ في تحديث ملف التقييم (Ground Truth): {e}")

Appending to services.py


In [9]:
%%writefile -a services.py
def sync_sent_emails_to_history():
    """
    مزامنة البريد المرسل:
    تقوم بقراءة البريد الذي قام المدير بإرساله فعلياً (سواء كان تعديلاً على مسودة البوت أو رداً يدوياً).
    إذا كان الرد مرتبطاً بطلب HR سابق، سيتم استنتاج قاعدة جديدة وحفظها.
    """
    processed_file = "processed_sent_ids.txt"
    processed_ids = set()
    if os.path.exists(processed_file):
        with open(processed_file, 'r', encoding='utf-8') as f:
            processed_ids = set(f.read().splitlines())

    updates_made = False 

    try:
        history_file = "email_history.txt"
        history_content = ""
        if os.path.exists(history_file):
            with open(history_file, 'r', encoding='utf-8') as f:
                history_content = f.read()

        mail = imaplib.IMAP4_SSL(IMAP_SERVER)
        mail.login(GMAIL_EMAIL, APP_PASSWORD)
        
        sent_folder = None
        for folder in ['"[Gmail]/Sent Mail"', '"[Gmail]/Sent"', 'Sent', 'البريد المرسل']:
            try:
                status, _ = mail.select(folder)
                if status == 'OK':
                    sent_folder = folder
                    break
            except: continue
        
        if not sent_folder: return False 

        # فحص آخر 10 رسائل مرسلة
        status, messages = mail.search(None, 'ALL')
        if status != 'OK': return False
        sent_msg_ids = messages[0].split()[-10:]
        
        with open(processed_file, 'a', encoding='utf-8') as f_ids:
            for msg_num in sent_msg_ids:
                _, data = mail.fetch(msg_num, "(RFC822)")
                msg = email.message_from_bytes(data[0][1])
                unique_id = msg.get("Message-ID", "").strip()
                
                if unique_id in processed_ids: continue 

                # استخراج النص المرسل (الرد النهائي المعتمد)
                sent_body = ""
                if msg.is_multipart():
                    for part in msg.walk():
                        if part.get_content_type() == "text/plain":
                            sent_body = safe_decode(part.get_payload(decode=True))
                            break
                else:
                    sent_body = safe_decode(msg.get_payload(decode=True))
                
                if not sent_body: continue
                # =========================================================
                # نستخدم In-Reply-To لمعرفة الإيميل الأصلي وتحديث سطر الإكسل الخاص به
                ref_id_for_excel = msg.get("In-Reply-To", "").strip()
                if ref_id_for_excel:
                    update_evaluation_ground_truth(ref_id_for_excel, sent_body)
                # =========================================================
                # الربط بالرسالة الأصلية عبر In-Reply-To أو References
                references = []
                if msg["In-Reply-To"]: references.append(msg["In-Reply-To"].strip())
                if msg["References"]: references.extend(msg["References"].split())
                
                found_original = False
                original_text = ""
                original_intent = None

                # البحث في السجل المحلي عن الرسالة الأصلية
                if references and history_content:
                    blocks = history_content.split("إيميل وارد بتاريخ")
                    for block in blocks:
                        for ref in references:
                            if ref in block:
                                found_original = True
                                original_text = block
                                # استخراج النية المسجلة
                                intent_match = re.search(r"Org-Intent:\s*(\w+)", block)
                                if intent_match:
                                    original_intent = intent_match.group(1)
                                break
                        if found_original: break
                
                # --- نقطة التعلم الجوهرية ---
                if found_original and original_intent == "HR":
                    print(f"🎓 تم اكتشاف رد بشري على معاملة HR (ID: {unique_id})... جاري التعلم.")
                    
                    # استنتاج القاعدة من الفرق بين الطلب والرد الفعلي
                    new_rule = deduce_administrative_rule(original_text, sent_body, "HR")
                    
                    if new_rule:
                        # حفظ القاعدة في ملف HR_rules.txt ليتم استرجاعها لاحقاً عبر FAISS
                        update_intent_rules_file("HR", new_rule)
                        updates_made = True
                
                # تسجيل المعرف لعدم معالجته مرة أخرى
                processed_ids.add(unique_id)
                f_ids.write(unique_id + "\n")

        mail.logout()
        return updates_made

    except Exception as e:
        print(f"⚠ خطأ في مزامنة المرسل: {e}")
        return False

Appending to services.py
